In [1]:
# Google Colab, Python 3.x 기준으로 작성한 노트북입니다.
# 이 셀은 전체 노트북에서 사용할 라이브러리를 한곳에 모아 둡니다.
# 과제에서 허용된 scikit-learn, pandas, numpy, joblib만 사용합니다.

from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import VotingClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import matthews_corrcoef, make_scorer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC


In [2]:
# Colab에서 실행할 때는 Google Drive를 마운트해야 파일이 세션 종료 후에도 남습니다.
# 로컬 Jupyter 또는 VS Code에서 실행하는 경우 google.colab 모듈이 없으므로 예외 처리로 건너뜁니다.

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive mount skipped:", type(exc).__name__)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# 실험 재현성을 위해 랜덤 시드를 하나로 고정합니다.
# random_state를 지원하는 StratifiedKFold, LinearSVC에 같은 값을 사용합니다.
RANDOM_STATE = 42

# 최종 제출 전에는 True로 두고 5-fold CV 결과를 남깁니다.
# 로컬에서 빠르게 셀 순서만 확인할 때는 False로 바꾸면 시간을 줄일 수 있습니다.
RUN_CV = True

# public_train.csv와 public_test.csv가 들어 있을 수 있는 위치를 순서대로 확인합니다.
# Colab에서는 보통 /content/drive/MyDrive/UD_26에 두면 됩니다.
# 만약 Drive 안에 mid 폴더까지 같이 올렸다면 /content/drive/MyDrive/UD_26/mid도 자동으로 찾습니다.
# 로컬에서는 이 노트북이 있는 저장소의 mid 폴더를 사용합니다.
DATA_DIR_CANDIDATES = [
    Path("/content/drive/MyDrive/UD_26"),
    Path("/content/drive/MyDrive/UD_26/mid"),
    Path("mid"),
    Path("."),
]

DATA_DIR = None
for candidate_dir in DATA_DIR_CANDIDATES:
    train_file = candidate_dir / "public_train.csv"
    test_file = candidate_dir / "public_test.csv"

    if train_file.exists() and test_file.exists():
        DATA_DIR = candidate_dir
        break

# 파일을 못 찾았을 때 현재 폴더로 조용히 넘어가지 않고, 확인해야 할 경로를 바로 보여줍니다.
# 이 에러가 나면 public_train.csv와 public_test.csv를 아래 후보 경로 중 하나에 넣으면 됩니다.
if DATA_DIR is None:
    checked_paths = "\n".join(
        f"- {candidate_dir / 'public_train.csv'} and {candidate_dir / 'public_test.csv'}"
        for candidate_dir in DATA_DIR_CANDIDATES
    )
    raise FileNotFoundError(
        "public_train.csv와 public_test.csv를 찾지 못했습니다.\n"
        "다음 경로 중 한 곳에 두었는지 확인하세요:\n"
        f"{checked_paths}"
    )

# submission CSV is saved inside MODEL_DIR with the model artifacts.
# 모델 pkl 파일은 여러 실험을 비교하기 쉽도록 model/노트북이름 폴더에 따로 모읍니다.
# 로컬에서는 c:/Users/user/Documents/Projects_src/UD_26/mid/model/VotingEnsemble_2LinearSVC_Ridge_CompNB 위치가 됩니다.
OUTPUT_DIR = DATA_DIR
NOTEBOOK_TAG = "VotingEnsemble_2LinearSVC_Ridge_CompNB"
MODEL_DIR = DATA_DIR / "model" / NOTEBOOK_TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "public_train.csv"
TEST_PATH = DATA_DIR / "public_test.csv"

# 모델 파일명에는 알고리즘, 벡터화 방식, 앙상블 방식, 랜덤 시드를 드러내 둡니다.
EXPERIMENT_TAG = "voting_linearsvc_ridge_complementnb_char_charwb_word_tfidf_hardvote_rs42"
MODEL_PATH = MODEL_DIR / f"{EXPERIMENT_TAG}.pkl"
FINAL_MODEL_PATH = MODEL_DIR / "final_pipeline.pkl"
SUBMISSION_PATH = MODEL_DIR / "submission.csv"

print("DATA_DIR:", DATA_DIR.resolve())
print("TRAIN_PATH:", TRAIN_PATH.resolve())
print("TEST_PATH:", TEST_PATH.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("MODEL_DIR:", MODEL_DIR.resolve())


DATA_DIR: /content/drive/MyDrive/UD_26
TRAIN_PATH: /content/drive/MyDrive/UD_26/public_train.csv
TEST_PATH: /content/drive/MyDrive/UD_26/public_test.csv
OUTPUT_DIR: /content/drive/MyDrive/UD_26
MODEL_DIR: /content/drive/MyDrive/UD_26/model/VotingEnsemble_2LinearSVC_Ridge_CompNB


In [4]:
# 학습 데이터와 테스트 데이터를 불러옵니다.
# public_train.csv에는 정답 라벨이 있고, public_test.csv에는 예측해야 할 text만 있습니다.
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

# 제출 전 기본 스키마를 먼저 확인합니다.
# 컬럼명이 다르면 이후 셀이 조용히 틀린 결과를 만들 수 있으므로 assert로 바로 멈추게 합니다.
print("train shape:", train.shape)
print("test shape:", test.shape)
print("train columns:", train.columns.tolist())
print("test columns:", test.columns.tolist())
print("label distribution:")
print(train["label"].value_counts())

# 과제에서 요구한 라벨 문자열 POSITIVE / NEGATIVE를 그대로 사용합니다.
# 0/1로 직접 바꾸면 inverse mapping 실수 위험이 있어 여기서는 문자열 라벨을 유지합니다.
assert {"row_id", "text", "label"}.issubset(train.columns)
assert {"row_id", "text"}.issubset(test.columns)
assert set(train["label"].unique()) <= {"POSITIVE", "NEGATIVE"}
assert train["text"].notna().all()
assert test["text"].notna().all()


train shape: (149995, 3)
test shape: (49997, 2)
train columns: ['row_id', 'text', 'label']
test columns: ['row_id', 'text']
label distribution:
label
NEGATIVE    75170
POSITIVE    74825
Name: count, dtype: int64


In [5]:
# 모델 입력으로 사용할 텍스트와 라벨을 분리합니다.
# 별도 검증 분할 없이, 성능 확인은 5-fold CV로 진행합니다.
# astype(str)은 혹시 모를 숫자형/결측 처리 문제를 줄이기 위한 방어적 변환입니다.
X = train["text"].astype(str)
y = train["label"].astype(str)

print("train rows:", X.shape[0])
print("label distribution:")
print(y.value_counts())


train rows: 149995
label distribution:
label
NEGATIVE    75170
POSITIVE    74825
Name: count, dtype: int64


In [6]:
# 텍스트를 여러 관점의 sparse feature로 바꿉니다.
# char n-gram은 띄어쓰기와 오탈자에 비교적 강하고, 한국어 짧은 문장에도 잘 맞습니다.
# char_wb n-gram은 단어 경계 주변의 문자 패턴을 더 안정적으로 잡습니다.
# word n-gram은 명확한 단어 조합 신호를 보완하기 위해 함께 사용합니다.
text_features = FeatureUnion([
    ("char", CountVectorizer(
        analyzer="char",
        ngram_range=(2, 5),
        min_df=2,
        max_features=400_000,
        lowercase=True,
        dtype=np.float32,
    )),
    ("char_wb", CountVectorizer(
        analyzer="char_wb",
        ngram_range=(2, 5),
        min_df=2,
        max_features=300_000,
        lowercase=True,
        dtype=np.float32,
    )),
    ("word", CountVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_features=200_000,
        token_pattern=r"(?u)\b\w+\b",
        lowercase=True,
        dtype=np.float32,
    )),
])

# 서로 다른 선형 계열 분류기를 hard voting으로 묶습니다.
# LinearSVC는 고차원 sparse 텍스트 분류에 강하고, RidgeClassifier는 안정적인 선형 기준선을 제공합니다.
# ComplementNB는 단어/문자 빈도 기반 분류에서 종종 좋은 보완 신호를 줍니다.
classifier = VotingClassifier(
    estimators=[
        ("linearsvc_c028", LinearSVC(
            C=0.28,
            random_state=RANDOM_STATE,
            dual="auto",
            max_iter=3000,
        )),
        ("ridge_alpha15", RidgeClassifier(
            alpha=1.5,
        )),
        ("complementnb_alpha001", ComplementNB(
            alpha=0.01,
        )),
        ("linearsvc_c040", LinearSVC(
            C=0.40,
            random_state=RANDOM_STATE,
            dual="auto",
            max_iter=3000,
        )),
    ],
    voting="hard",
    weights=[2.5, 4.0, 8.0, 2.0],
)

# 최종 모델은 반드시 하나의 Pipeline으로 구성합니다.
# 이렇게 저장하면 원문 text를 넣었을 때 벡터화부터 예측까지 한 번에 재현됩니다.
pipeline = Pipeline([
    ("features", text_features),
    ("tfidf", TfidfTransformer(
        sublinear_tf=True,
        norm="l2",
        use_idf=True,
        smooth_idf=True,
    )),
    ("classifier", classifier),
])

pipeline


Pipeline(steps=[('features',
                 FeatureUnion(transformer_list=[('char',
                                                 CountVectorizer(analyzer='char',
                                                                 dtype=<class 'numpy.float32'>,
                                                                 max_features=400000,
                                                                 min_df=2,
                                                                 ngram_range=(2,
                                                                              5))),
                                                ('char_wb',
                                                 CountVectorizer(analyzer='char_wb',
                                                                 dtype=<class 'numpy.float32'>,
                                                                 max_features=300000,
                                                                 min_df=2,
                                                                 ngram_range=(2,
                                                                              5))),
                                                ('word',
                                                 CountVectorizer(dtype=<class 'numpy.float3...
                                                                 token_pattern='(?u)\\b\\w+\\b'))])),
                ('tfidf', TfidfTransformer(sublinear_tf=True)),
                ('classifier',
                 VotingClassifier(estimators=[('linearsvc_c028',
                                               LinearSVC(C=0.28, max_iter=3000,
                                                         random_state=42)),
                                              ('ridge_alpha15',
                                               RidgeClassifier(alpha=1.5)),
                                              ('complementnb_alpha001',
                                               ComplementNB(alpha=0.01)),
                                              ('linearsvc_c040',
                                               LinearSVC(C=0.4, max_iter=3000,
                                                         random_state=42))],
                                  weights=[2.5, 4.0, 8.0, 2.0]))])

In [7]:
# 과제 요구에 맞게 전체 public_train.csv를 대상으로 5-fold cross-validation을 수행합니다.
# 샘플링으로 줄이지 않고 전체 데이터를 사용해야 최종 노트북 설명이 일관됩니다.
if RUN_CV:
    mcc_scorer = make_scorer(matthews_corrcoef)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    # n_jobs=1은 Colab/Windows 환경에서 메모리 사용량을 예측하기 쉽도록 둔 설정입니다.
    # 충분한 메모리가 있으면 n_jobs=-1로 바꿔 병렬 실행할 수 있습니다.
    cv_scores = cross_val_score(
        pipeline,
        X,
        y,
        scoring=mcc_scorer,
        cv=cv,
        n_jobs=1,
    )

    print("5-fold CV MCC scores:", cv_scores)
    print("5-fold CV MCC mean:", cv_scores.mean())
    print("5-fold CV MCC std:", cv_scores.std())
else:
    print("RUN_CV=False: skipped 5-fold CV.")


5-fold CV MCC scores: [0.75523161 0.74719154 0.75594784 0.75746655 0.75538872]
5-fold CV MCC mean: 0.7542452507623362
5-fold CV MCC std: 0.0036141632475958714


In [8]:
# 5-fold CV 확인이 끝나면 전체 학습 데이터로 최종 모델을 학습합니다.
# 제출용 모델은 public_train.csv 전체를 사용해 다시 fit합니다.
final_pipeline = Pipeline([
    ("features", text_features),
    ("tfidf", TfidfTransformer(
        sublinear_tf=True,
        norm="l2",
        use_idf=True,
        smooth_idf=True,
    )),
    ("classifier", classifier),
])

final_pipeline.fit(X, y)

# 실험 파일명 모델과 과제 필수 파일명 final_pipeline.pkl을 둘 다 MODEL_DIR에 저장합니다.
# 여러 앙상블 방법을 시도할 때도 model 폴더 아래에 노트북별 하위 폴더를 만들어 pkl을 비교할 수 있습니다.
# LMS 제출에는 final_pipeline.pkl 이름이 필요하고, 실험 파일은 어떤 설정이었는지 추적하기 위한 파일입니다.
joblib.dump(final_pipeline, MODEL_PATH)
joblib.dump(final_pipeline, FINAL_MODEL_PATH)

print("saved experiment model:", MODEL_PATH)
print("saved final model:", FINAL_MODEL_PATH)



saved experiment model: /content/drive/MyDrive/UD_26/model/VotingEnsemble_2LinearSVC_Ridge_CompNB/voting_linearsvc_ridge_complementnb_char_charwb_word_tfidf_hardvote_rs42.pkl
saved final model: /content/drive/MyDrive/UD_26/model/VotingEnsemble_2LinearSVC_Ridge_CompNB/final_pipeline.pkl


In [9]:
# public_test.csv의 text만 사용해 예측합니다.
# hidden label은 없으며, train 행을 섞어 쓰지 않도록 test 데이터프레임에서만 row_id와 text를 가져옵니다.
test_pred = final_pipeline.predict(test["text"].astype(str))

# 제출 파일은 정확히 row_id, pred_label 두 컬럼만 가져야 합니다.
# row_id 순서는 public_test.csv의 순서를 그대로 유지합니다.
submission = test[["row_id"]].copy()
submission["pred_label"] = test_pred

# 제출 전 필수 조건을 assert로 확인합니다.
# 여기서 실패하면 CSV를 업로드하기 전에 원인을 먼저 확인해야 합니다.
assert len(submission) == len(test)
assert submission.columns.tolist() == ["row_id", "pred_label"]
assert submission["row_id"].equals(test["row_id"])
assert set(submission["pred_label"].unique()) <= {"POSITIVE", "NEGATIVE"}

# UTF-8 CSV로 저장하고 index는 포함하지 않습니다.
submission.to_csv(SUBMISSION_PATH, index=False, encoding="utf-8")

print(submission.head())
print("Prediction distribution:")
print(submission["pred_label"].value_counts())
print("saved submission:", SUBMISSION_PATH)


        row_id pred_label
0  test-000001   POSITIVE
1  test-000002   POSITIVE
2  test-000003   NEGATIVE
3  test-000004   NEGATIVE
4  test-000005   NEGATIVE
Prediction distribution:
pred_label
NEGATIVE    25165
POSITIVE    24832
Name: count, dtype: int64
saved submission: /content/drive/MyDrive/UD_26/submission.csv


In [10]:
# 저장한 final_pipeline.pkl을 다시 불러와서 실제 제출 artifact가 정상인지 간단히 확인합니다.
# Pipeline 안에 vectorizer와 classifier가 모두 들어 있으므로 원문 text를 바로 predict할 수 있어야 합니다.
loaded_pipeline = joblib.load(FINAL_MODEL_PATH)
smoke_pred = loaded_pipeline.predict(test["text"].astype(str).head(5))

print("loaded model type:", type(loaded_pipeline))
print("smoke predictions:", smoke_pred)
assert set(smoke_pred) <= {"POSITIVE", "NEGATIVE"}


loaded model type: <class 'sklearn.pipeline.Pipeline'>
smoke predictions: ['POSITIVE' 'POSITIVE' 'NEGATIVE' 'NEGATIVE' 'NEGATIVE']
